# Nexus Foundation Model Training (Alibaba ClusterData 2021)

This notebook downloads the Alibaba Microservice traces, preprocesses the data to extract normalized QPS, Latency, and resource utilization, and trains a universal LSTM Foundation Model for the Predictive Pod Autoscaler (PPA).

## 1. Environment Setup & Data Download

We download `MSRTQps` (QPS + response time) and `MSResource` (CPU + memory) to the `data/` folder.

**Why MSRTQps instead of MSCallGraph:** The `MSRTQps` table already provides per-service QPS (calls/sec) and response time (ms) — directly mappable to our Prometheus queries. `MSCallGraph` contains individual call traces that would need expensive aggregation.

Schema reference ([Alibaba ClusterData 2021 README](https://github.com/alibaba/clusterdata/tree/master/cluster-trace-microservices-v2021)):

**MSRTQps** (no header row, indices 0–4):
| Idx | Column       | Description |
|-----|-------------|-------------|
| 0   | timestamp    | 60s interval, range 0–43200000 **milliseconds** |
| 1   | msname       | Microservice name |
| 2   | msinstanceid | Container/instance ID |
| 3   | metrics      | e.g. `HTTP_MCR` (calls/sec), `HTTP_RT` (ms) |
| 4   | value        | Calls/sec for `_MCR`; ms for `_RT` |

**MSResource** (no header row, indices 0–5):
| Idx | Column              | Description |
|-----|---------------------|-------------|
| 0   | timestamp           | 60s interval, milliseconds |
| 1   | msname              | Microservice name |
| 2   | msinstanceid        | Instance ID |
| 3   | nodeid              | Bare-metal node ID |
| 4   | cpu_utilization     | CPU utilization |
| 5   | memory_utilization  | Memory utilization |

In [ ]:
!pip install pandas numpy scikit-learn tensorflow matplotlib pyarrow
import os
import gc
import tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

os.makedirs("data", exist_ok=True)

In [ ]:
BASE_URL = "http://aliopentrace.oss-cn-beijing.aliyuncs.com/v2021MicroservicesTraces"

def download_and_extract(url, dest_dir="data"):
    filename = url.split("/")[-1]
    tar_path = os.path.join(dest_dir, filename)

    if not os.path.exists(tar_path):
        print(f"Downloading {filename}...")
        try:
            urllib.request.urlretrieve(url, tar_path)
        except Exception as e:
            print(f"Failed to download: {e}")
            return None

    print(f"Extracting {filename}...")
    try:
        with tarfile.open(tar_path, "r:gz") as tar:
            if hasattr(tarfile, 'data_filter'):
                tar.extractall(path=dest_dir, filter='data')
            else:
                tar.extractall(path=dest_dir)
            csv_name = tar.getnames()[0]
            return os.path.join(dest_dir, csv_name)
    except Exception as e:
        print(f"Extraction failed: {e}")
        return None

# Download a small sample (2 chunks each)
# Full dataset: MSRTQps has 25 files (0-24), MSResource has 12 files (0-11)
NUM_CHUNKS = 2

rtqps_csvs = []
for i in range(NUM_CHUNKS):
    url = f"{BASE_URL}/MSRTQps/MSRTQps_{i}.tar.gz"
    csv_path = download_and_extract(url)
    if csv_path:
        rtqps_csvs.append(csv_path)

resource_csvs = []
for i in range(NUM_CHUNKS):
    url = f"{BASE_URL}/MSResource/MSResource_{i}.tar.gz"
    csv_path = download_and_extract(url)
    if csv_path:
        resource_csvs.append(csv_path)

## 2. Data Loading & Preprocessing

### MSRTQps → QPS & Latency

The `MSRTQps` table has a `metrics` column distinguishing call rate (`_MCR`) vs response time (`_RT`). We pivot on this to get per-service-per-minute QPS and P95 latency.

**Key fix:** Timestamps are in **milliseconds** (0–43,200,000 = 12 hours), NOT microseconds.

In [ ]:
# MSRTQps schema (NO header row): timestamp(0), msname(1), msinstanceid(2), metrics(3), value(4)
# We only need: timestamp(0), msname(1), metrics(3), value(4)

MCR_METRICS = ['consumerRPC_MCR', 'providerRPC_MCR', 'HTTP_MCR', 'providerMQ_MCR', 'consumerMQ_MCR']
RT_METRICS  = ['consumerRPC_RT', 'providerRPC_RT', 'HTTP_RT', 'providerMQ_RT', 'consumerMQ_RT']

MS_TO_MIN = 60_000  # 1 minute in milliseconds

aggregated_traffic = []

for csv_file in rtqps_csvs:
    print(f"Processing {csv_file}...")
    try:
        # No header row — use positional indices
        df = pd.read_csv(csv_file, header=None, usecols=[0, 1, 3, 4],
                         names=['timestamp', 'msname', 'metrics', 'value'],
                         low_memory=False)

        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df = df.dropna(subset=['timestamp', 'msname', 'value'])

        # Bin to minutes: timestamps are in MILLISECONDS
        df['minute'] = (df['timestamp'] // MS_TO_MIN).astype(int)

        # Split into QPS rows (_MCR = calls/sec) and RT rows (_RT = response time ms)
        df_mcr = df[df['metrics'].isin(MCR_METRICS)]
        df_rt  = df[df['metrics'].isin(RT_METRICS)]

        # Aggregate QPS: sum calls/sec across all instances per service per minute
        qps_agg = df_mcr.groupby(['minute', 'msname']).agg(
            requests_per_second=('value', 'sum')
        ).reset_index()

        # Aggregate RT: P95 of response times per service per minute
        rt_agg = df_rt.groupby(['minute', 'msname']).agg(
            latency_p95_ms=('value', lambda x: np.percentile(x, 95))
        ).reset_index()

        # Merge QPS and RT on (minute, msname)
        merged = pd.merge(qps_agg, rt_agg, on=['minute', 'msname'], how='outer')
        aggregated_traffic.append(merged)

        del df, df_mcr, df_rt
        gc.collect()
        print(f"  Aggregated {len(merged)} minute-service rows")
    except Exception as e:
        print(f"Error processing {csv_file}: {e}")

if aggregated_traffic:
    df_traffic = pd.concat(aggregated_traffic).groupby(['minute', 'msname']).agg(
        requests_per_second=('requests_per_second', 'sum'),
        latency_p95_ms=('latency_p95_ms', 'mean')
    ).reset_index()
    df_traffic.rename(columns={'msname': 'service'}, inplace=True)
    # Fill NaN from outer merge (services with QPS but no RT, or vice versa)
    df_traffic['latency_p95_ms'] = df_traffic['latency_p95_ms'].fillna(0)
    df_traffic['requests_per_second'] = df_traffic['requests_per_second'].fillna(0)
    print(f"Traffic data shape: {df_traffic.shape}")
    display(df_traffic.head())
else:
    raise RuntimeError("No MSRTQps data loaded. Check download/extraction.")

### MSResource → CPU & Memory

Schema (no header row): `timestamp(0), msname(1), msinstanceid(2), nodeid(3), cpu_utilization(4), memory_utilization(5)`

We need: `timestamp(0), msname(1), cpu_utilization(4), memory_utilization(5)`

In [ ]:
# MSResource schema (NO header row): timestamp(0), msname(1), msinstanceid(2), nodeid(3), cpu_utilization(4), memory_utilization(5)
# We need: timestamp(0), msname(1), cpu_utilization(4), memory_utilization(5)

aggregated_metrics = []

for csv_file in resource_csvs:
    print(f"Processing {csv_file}...")
    try:
        # No header row — positional indices. msname is index 1 (not 3)
        df = pd.read_csv(csv_file, header=None, usecols=[0, 1, 4, 5],
                         names=['timestamp', 'msname', 'cpu_utilization', 'memory_utilization'],
                         low_memory=False)

        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
        df['cpu_utilization'] = pd.to_numeric(df['cpu_utilization'], errors='coerce')
        df['memory_utilization'] = pd.to_numeric(df['memory_utilization'], errors='coerce')
        df = df.dropna(subset=['timestamp', 'msname'])

        # Bin to minutes: timestamps in MILLISECONDS
        df['minute'] = (df['timestamp'] // MS_TO_MIN).astype(int)

        # Aggregate: mean across all instances per service per minute
        agg = df.groupby(['minute', 'msname']).agg(
            cpu_utilization_pct=('cpu_utilization', 'mean'),
            memory_utilization_pct=('memory_utilization', 'mean')
        ).reset_index()
        aggregated_metrics.append(agg)

        del df
        gc.collect()
        print(f"  Aggregated {len(agg)} minute-service rows")
    except Exception as e:
        print(f"Error processing {csv_file}: {e}")

if aggregated_metrics:
    df_metrics = pd.concat(aggregated_metrics).groupby(['minute', 'msname']).mean().reset_index()
    df_metrics.rename(columns={'msname': 'service'}, inplace=True)
    print(f"Metrics data shape: {df_metrics.shape}")

    # Merge Traffic and Metrics on minute & service
    df = pd.merge(df_traffic, df_metrics, on=['minute', 'service'], how='inner')
    df['timestamp'] = pd.to_datetime(df['minute'] * MS_TO_MIN, unit='ms')
    df['error_rate'] = 0.0  # Not available in Alibaba traces; zeroed as baseline

    # Drop rows with missing core metrics
    df = df.dropna(subset=['requests_per_second', 'cpu_utilization_pct', 'memory_utilization_pct'])
    df = df.reset_index(drop=True)
    print(f"Merged data shape: {df.shape}")
    display(df.head())
else:
    raise RuntimeError("No MSResource data loaded. Check download/extraction.")

## 3. Feature Engineering & Normalization
To train a universal Foundation Model, we must **normalize RPS** and calculate target multipliers. We will scale `requests_per_second` by the trailing rolling maximum per service to get `normalized_rps` (0 to 1).

In [ ]:
def engineer_features(df, window_size=60):
    dfs = []
    for svc, group in df.groupby('service'):
        group = group.sort_values('minute').copy()
        if len(group) < window_size:
            continue  # skip services with too few data points

        # Trailing rolling max RPS for normalization
        group['rolling_max_rps'] = group['requests_per_second'].rolling(window=window_size, min_periods=1).max()
        group['rolling_max_rps'] = np.maximum(group['rolling_max_rps'], 1.0)  # prevent div-by-zero

        # Scale-invariant feature: normalized RPS (0 to 1)
        group['normalized_rps'] = group['requests_per_second'] / group['rolling_max_rps']

        # Normalized latency relative to rolling max
        group['rolling_max_latency'] = group['latency_p95_ms'].rolling(window=window_size, min_periods=1).max()
        group['rolling_max_latency'] = np.maximum(group['rolling_max_latency'], 1.0)
        group['latency_normalized'] = group['latency_p95_ms'] / group['rolling_max_latency']

        # Accelerations (derivatives)
        group['rps_acceleration'] = group['normalized_rps'].diff()
        group['cpu_acceleration'] = group['cpu_utilization_pct'].diff()

        # Target: Normalized RPS at t+3 minutes
        group['target_normalized_rps_t3m'] = group['normalized_rps'].shift(-3)

        dfs.append(group)

    if not dfs:
        return pd.DataFrame()
    res = pd.concat(dfs).dropna().reset_index(drop=True)

    # Temporal features from timestamp
    res['hour_sin'] = np.sin(2 * np.pi * res['timestamp'].dt.hour / 24)
    res['hour_cos'] = np.cos(2 * np.pi * res['timestamp'].dt.hour / 24)
    res['dow_sin'] = np.sin(2 * np.pi * res['timestamp'].dt.dayofweek / 7)
    res['dow_cos'] = np.cos(2 * np.pi * res['timestamp'].dt.dayofweek / 7)
    res['is_weekend'] = res['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

    return res

print("Engineering scale-invariant features...")
df_features = engineer_features(df)
if len(df_features) == 0:
    raise RuntimeError("No data after feature engineering. Check that services have >= 60 minutes of data.")

display(df_features[['service', 'requests_per_second', 'rolling_max_rps', 'normalized_rps', 'target_normalized_rps_t3m']].head())

## 4. Sliding Window & Train/Test Split
We create overlapping windows of size `LOOKBACK_STEPS` to feed into the LSTM.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

LOOKBACK_STEPS = 60
FEATURE_COLS = [
    'normalized_rps', 'cpu_utilization_pct', 'memory_utilization_pct',
    'latency_normalized', 'error_rate', 'rps_acceleration', 'cpu_acceleration',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend'
]
TARGET_COL = 'target_normalized_rps_t3m'

# Scale universal features (cpu, mem, latency-derived) to [0, 1]
scaler = MinMaxScaler()
df_features[FEATURE_COLS] = scaler.fit_transform(df_features[FEATURE_COLS])

# Save the *feature* scaler so the operator can reproduce the same input scale
import joblib
joblib.dump(scaler, 'foundation_scaler.pkl')
print(f"Saved feature scaler: foundation_scaler.pkl  (n_features={len(FEATURE_COLS)})")

def create_sequences(df, feature_cols, target_col, lookback):
    X, y = [], []
    for svc, group in df.groupby('service'):
        feat_vals = group[feature_cols].values
        target_vals = group[target_col].values
        for i in range(len(feat_vals) - lookback):
            X.append(feat_vals[i:(i + lookback)])
            y.append(target_vals[i + lookback - 1])
    return np.array(X), np.array(y)

print("Creating sequences...")
X, y = create_sequences(df_features, FEATURE_COLS, TARGET_COL, LOOKBACK_STEPS)

# Shuffle sequences globally
indices = np.arange(len(X))
np.random.shuffle(indices)
X, y = X[indices], y[indices]

# 80/10/10 Split
n = len(X)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

# Match the model's two outputs: [demand, attention_weights (dummy zeros)]
y_train_dummy = np.zeros((len(y_train), LOOKBACK_STEPS), dtype=np.float32)
y_val_dummy   = np.zeros((len(y_val),   LOOKBACK_STEPS), dtype=np.float32)
y_test_dummy  = np.zeros((len(y_test),  LOOKBACK_STEPS), dtype=np.float32)
y_train_out = {"normalized_demand_output": y_train, "attention_weights": y_train_dummy}
y_val_out   = {"normalized_demand_output": y_val,   "attention_weights": y_val_dummy}
y_test_out  = {"normalized_demand_output": y_test,  "attention_weights": y_test_dummy}

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 5. Foundation Model Architecture — Residual Self-Attention LSTM

The encoder is a **self-attention augmented LSTM** with a **residual connection** between LSTM outputs and post-attention features, plus **batch normalization** in the dense head. Loss is **Huber** instead of pure MSE — robust to the long-tail of bursty Alibaba traffic without becoming a MAE-only-regressor at the small-error regime.

**Block layout:**

```
Input window (60 × 12)
  │
  ▼
BatchNorm   ← normalises the 12-feature input (mean over time-batch)
  │
  ▼
LSTM(128, return_sequences=True, unroll=True)
  │
  ▼
Dropout(0.2)
  │
  ▼
LSTM(64, return_sequences=True, unroll=True)
  │
  ▼
Self-Attention over time (QKV via Bahdanau scoring → softmax weights)
  │
  ▼
Add & LayerNorm  ← residual: x + attention(x), then normalized
  │
  ▼
Flatten context  ← (batch, time, feat) → (batch, time*feat) for the dense head
  │
  ▼
Dense(128, ReLU) → BatchNorm → Dropout(0.3)
  │
  ▼
Dense(64, ReLU)
  │
  ▼
Dense(1, linear) — 'normalized_demand_output'
```

The auxiliary output 'attention_weights' is kept in the graph for **fine-tuning**, so per-app retraining can use it as a learned-prior regularizer. The TFLite export strips it back to a single demand tensor.

**Loss:** `Huber(δ=0.05)` — squared for residuals below the 5% normalized-RPS threshold, linear beyond. Stops single minute-spike errors from squaring-out the gradients, while still penalising calibrated-scale mistakes.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, backend as K, Model

# ---------- Huber loss --------------------------------------------------------
# delta=0.05 ≈ 5% normalized-RPS. Below that we learn as MSE; above that the loss
# becomes L1, so a single minute-spike (e.g. 0 → 1 on a previously idle service)
# stops sqrt-exploding the gradient and wrecking the LSTM state.
def huber_loss(delta: float = 0.05):
    delta = tf.constant(delta, dtype=tf.float32)

    def _huber(y_true, y_pred):
        residual = tf.abs(y_true - y_pred)
        squared = K.minimum(residual, delta)
        linear = residual - squared
        return 0.5 * K.mean(K.square(squared), axis=-1) + delta * K.mean(linear, axis=-1)

    return _huber


# ---------- Attention ---------------------------------------------------------
class SelfAttentionLayer(layers.Layer):
    """Bahdanau-style additive self-attention over the LSTM time axis.

    Returns:
        context: (batch, features) softmax-weighted sum of `inputs`
        weights: (batch, time, 1)  attention map (exposed for fine-tuning)
    """
    def __init__(self, attention_units: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.attention_units = attention_units
        self.W = layers.Dense(attention_units, activation="tanh", name="att_W")
        self.u = layers.Dense(1, use_bias=False, name="att_u")
        self.layernorm = layers.LayerNormalization(epsilon=1e-6, name="att_layernorm")

    def call(self, inputs):
        # inputs: (batch, time, features)
        score = self.u(self.W(inputs))            # (batch, time, 1)
        weights = tf.nn.softmax(score, axis=1)    # (batch, time, 1)
        context = tf.reduce_sum(weights * inputs, axis=1)  # (batch, features)
        # Residual: x + attention(x), then LayerNorm
        # Project attention output back to the time axis so the residual matches shape
        attended_time = weights * inputs          # (batch, time, features)
        residual = self.layernorm(inputs + attended_time)
        # Reduced representation for downstream dense layers
        residual_context = tf.reduce_sum(residual, axis=1)  # (batch, features)
        return residual_context, weights

    def get_config(self):
        config = super().get_config()
        config.update({"attention_units": self.attention_units})
        return config


def build_foundation_model(lookback, num_features, attention_units=64, huber_delta=0.05):
    """Residual self-attention LSTM with batch-normalised dense head."""
    inputs = layers.Input(shape=(lookback, num_features), name="input_window")

    # Input-side BatchNorm — stabilises LSTM gradients against the wide
    # spread of normalized_RPS/normalized_latency distributions per service.
    x = layers.BatchNormalization(name="input_bn")(inputs)

    # LSTM encoder
    x = layers.LSTM(128, return_sequences=True, unroll=True, name="lstm_1")(x)
    x = layers.Dropout(0.2, name="dropout_1")(x)
    lstm_seq = layers.LSTM(64, return_sequences=True, unroll=True, name="lstm_2")(x)
    x = layers.Dropout(0.2, name="dropout_2")(lstm_seq)

    # Self-attention with residual + LayerNorm
    context, att_w = SelfAttentionLayer(attention_units, name="self_attention")(x)

    # Dense head with batch-normalised activations
    h = layers.Dense(128, activation="relu", name="dense_1")(context)
    h = layers.BatchNormalization(name="dense_1_bn")(h)
    h = layers.Dropout(0.3, name="dense_1_dropout")(h)
    h = layers.Dense(64, activation="relu", name="dense_2")(h)
    h = layers.BatchNormalization(name="dense_2_bn")(h)
    demand = layers.Dense(1, activation="linear", name="normalized_demand_output")(h)

    attention_output = layers.Lambda(
        lambda w: tf.squeeze(w, axis=-1),
        name="attention_weights",
    )(att_w)

    model = Model(
        inputs=inputs,
        outputs=[demand, attention_output],
        name="ppa_foundation_model",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
        loss={
            "normalized_demand_output": huber_loss(huber_delta),
            "attention_weights": lambda y_t, y_p: 0.0,
        },
        loss_weights={"normalized_demand_output": 1.0, "attention_weights": 0.0},
        metrics={"normalized_demand_output": ["mae", "mse"]},
    )
    return model


model = build_foundation_model(LOOKBACK_STEPS, len(FEATURE_COLS))
model.summary()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_normalized_demand_output_loss",
    patience=10, min_delta=1e-4, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_normalized_demand_output_loss",
    factor=0.5, patience=5, min_lr=1e-5
)

print("Training Foundation Model (Residual Self-Attention LSTM, Huber Loss)...")
history = model.fit(
    X_train, y_train_out,
    validation_data=(X_val, y_val_out),
    epochs=50,
    batch_size=256,
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

# Track LR for convergence plot
history.history.setdefault("lr", [])
if not history.history["lr"]:
    history.history["lr"] = [1e-3] * len(history.history["loss"])

print(f"Best epoch: {early_stopping.best_epoch + 1}, "
      f"best val_loss: {early_stopping.best:.4f}")


## 7. Training Convergence & Forecast Quality

Before exporting, two diagnostics run on the trained weights:

1. **Convergence** – train/val Huber loss + MAE vs epoch, with the LR schedule overlaid so drops in LR are visible against the actual loss movement.
2. **Forecast inspection** – predicted vs actual `normalized_rps_t3m` on a 1000-sample test slice, plus an error distribution histogram (MAE + RMSE printed in the title).

These catch pathology-class failures (constant-output collapse, large bias, or attention not learning) before the TFLite export.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Foundation Model — Convergence & Forecast Quality", fontsize=14)

# 1 — Huber loss
ax = axes[0, 0]
ax.plot(history.history["loss"], label="train_loss (Huber)")
ax.plot(history.history["val_loss"], label="val_loss (Huber)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Huber Loss")
ax.legend(); ax.grid(alpha=0.3)

# 2 — MAE
ax = axes[0, 1]
ax.plot(history.history.get("normalized_demand_output_mae", []), label="train_MAE")
ax.plot(history.history.get("val_normalized_demand_output_mae", []), label="val_MAE")
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE"); ax.set_title("Normalized RPS MAE")
ax.legend(); ax.grid(alpha=0.3)

# 3 — Forecast: predicted vs actual on test slice
import random
import numpy as np

np.random.seed(0)
# Get predictions from the single-output inference model
inp = model.get_layer("input_window").input
dem = model.get_layer("normalized_demand_output").output
inf_model = Model(inp, dem)
inf_model.set_weights(model.get_weights())
y_pred = inf_model.predict(X_test[:1000], verbose=0).flatten()
y_true = y_test[:1000]

ax = axes[1, 0]
ax.plot(y_true, label="actual", alpha=0.7)
ax.plot(y_pred, label="predicted", alpha=0.7)
ax.set_xlabel("Sample (test slice)"); ax.set_ylabel("normalized_rps_t3m")
ax.set_title("Forecast vs Actual (1st 1000 test samples)")
ax.legend(); ax.grid(alpha=0.3)

# 4 — Error distribution
errors = y_pred - y_true
ax = axes[1, 1]
ax.hist(errors, bins=50, alpha=0.6, edgecolor="black")
ax.axvline(0, color="red", linestyle="--", alpha=0.5)
ax.set_xlabel("Prediction error"); ax.set_ylabel("Count")
ax.set_title(f"Error dist  (MAE={np.abs(errors).mean():.4f}, RMSE={np.sqrt((errors**2).mean()):.4f})")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("foundation_convergence.png", dpi=150)
plt.show()
print(f"Saved diagnostic plot: foundation_convergence.png")


## 8. Exporting to Keras + TFLite

Three artifacts are written:

| Filename | Format | Used by |
|----------|--------|---------|
| `foundation_model_v1.keras` | Full Keras model | per-app **fine-tuning** – `model.train.py --finetune-from` |
| `foundation_model_v1.tflite` | TFLite (single `normalized_demand_output`) | operator at runtime |
| `foundation_scaler.pkl` | joblib MinMaxScaler on 12 features | operator input scaling |

The exported TFLite model strips the auxiliary attention head, returning only the `(1,)` demand tensor.

In [ ]:
from tensorflow.keras import Model as _KerasModel

# Evaluate on test set (only the demand output has a real metric)
test_metrics = model.evaluate(X_test, y_test_out, verbose=0, return_dict=True)
print(f"Test: {test_metrics}")

# 1) Save full Keras model
keras_path = "foundation_model_v1.keras"
model.save(keras_path)
print(f"Saved Keras model: {keras_path}")

# 2) Single-output inference model for TFLite
inference_input = model.get_layer("input_window").input
demand_output = model.get_layer("normalized_demand_output").output
inference_model = _KerasModel(inputs=inference_input, outputs=demand_output)
inference_model.set_weights(model.get_weights())

# 3) Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
tflite_path = "foundation_model_v1.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print(f"Saved TFLite model:  {tflite_path}  ({len(tflite_model)/1024:.1f} KB)")

# 4) Verify TFLite inference matches Keras
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
in_d = interpreter.get_input_details()[0]
out_d = interpreter.get_output_details()[0]
print(f"  input shape={in_d['shape']}  output shape={out_d['shape']}")
interpreter.set_tensor(in_d["index"], X_test[:1].astype(np.float32))
interpreter.invoke()
tflite_pred = interpreter.get_tensor(out_d["index"])
keras_pred = inference_model.predict(X_test[:1], verbose=0)
print(f"  Keras={keras_pred[0,0]:.6f}  TFLite={tflite_pred[0,0]:.6f}  diff={abs(keras_pred[0,0]-tflite_pred[0,0]):.2e}")

print(f"\n3 files for operator:")
print(f"  - {tflite_path}     -> model.tflite  in bundle")
print(f"  - foundation_scaler.pkl  -> scaler.pkl  in bundle")
print(f"  - {keras_path}  -> --finetune-from")
